In [3]:
!pip install datasets pandas openpyxl

In [9]:
#import
from datasets import load_dataset
import pandas as pd

#dataset loading
pr_request = load_dataset("hao-li/AIDev", name="pull_request", split="train")

#dataframe
df_pr_request= pr_request.to_pandas()

print("Loaded Data:")
# print(df.head())
print("\nColumns:", df_pr_request.columns)


Loaded Data:

Columns: Index(['id', 'number', 'title', 'body', 'agent', 'user_id', 'user', 'state',
       'created_at', 'closed_at', 'merged_at', 'repo_id', 'repo_url',
       'html_url'],
      dtype='object')


In [7]:
#dataset loading
pr_commit_details = load_dataset("hao-li/AIDev", name="pr_commit_details", split="train")

#dataframe
df_pr_commit_details= pr_commit_details.to_pandas()

#prints
print("Columns of pr commit details:")
print(df_pr_commit_details.columns.tolist())

Columns of pr commit details:
['sha', 'pr_id', 'author', 'committer', 'message', 'commit_stats_total', 'commit_stats_additions', 'commit_stats_deletions', 'filename', 'status', 'additions', 'deletions', 'changes', 'patch']


In [10]:
merged = df_pr_commit_details.merge(
    df_pr_request,
    left_on="pr_id",
    right_on="id",
    how="inner",
    suffixes=("_commit", "_pr")
)

print("Merged rows:", len(merged))


Merged rows: 711923


In [12]:
TEST_FILE_PATTERNS = [
    r"/test/",
    r"/tests/",
    r"__tests__",
    r"_test\.py$",
    r"Test\.java$",
    r"\.spec\.js$",
    r"\.test\.js$"
]

def is_test_file(filename):
    if pd.isna(filename):
        return False
    return any(re.search(p, filename, re.IGNORECASE) for p in TEST_FILE_PATTERNS)

merged["is_test_file"] = merged["filename"].apply(is_test_file)


In [13]:
def count_loc(patch):
    if pd.isna(patch):
        return 0, 0
    added = sum(1 for l in patch.splitlines() if l.startswith("+") and not l.startswith("+++"))
    removed = sum(1 for l in patch.splitlines() if l.startswith("-") and not l.startswith("---"))
    return added, removed

merged[["loc_added", "loc_removed"]] = merged["patch"].apply(
    lambda x: pd.Series(count_loc(x))
)

merged["test_loc_added"] = merged["loc_added"] * merged["is_test_file"]
merged["test_loc_removed"] = merged["loc_removed"] * merged["is_test_file"]
merged["prod_loc_added"] = merged["loc_added"] * (~merged["is_test_file"])


In [14]:
coverage_proxy = merged.groupby("agent").agg(
    test_loc_added=("test_loc_added", "sum"),
    prod_loc_added=("prod_loc_added", "sum")
).reset_index()

coverage_proxy["test_to_code_ratio"] = (
    coverage_proxy["test_loc_added"] /
    coverage_proxy["prod_loc_added"].replace(0, 1)
)


In [16]:
TEST_TYPE_PATTERNS = {
    "unit": [r"unit", r"_test", r"Test\.java"],
    "integration": [r"integration", r"IT\.java", r"database", r"api"],
    "e2e": [r"e2e", r"cypress", r"playwright", r"selenium"],
    "performance": [r"benchmark", r"perf", r"jmh"]
}

def detect_test_type(text):
    if pd.isna(text):
        return None
    for t, patterns in TEST_TYPE_PATTERNS.items():
        if any(re.search(p, text, re.IGNORECASE) for p in patterns):
            return t
    return None

merged["test_type"] = merged.apply(
    lambda r: detect_test_type(str(r["filename"]) + " " + str(r["patch"])),
    axis=1
)


In [17]:
ASSERT_PATTERNS = [
    r"\bassert\b",
    r"assertEquals",
    r"assertTrue",
    r"assertFalse",
    r"assertThrows",
    r"expect\(",
    r"toBe\("
]

def count_assertions(patch):
    if pd.isna(patch):
        return 0
    return sum(len(re.findall(p, patch)) for p in ASSERT_PATTERNS)

merged["assertion_count"] = merged.apply(
    lambda r: count_assertions(r["patch"]) if r["is_test_file"] else 0,
    axis=1
)

In [18]:
rq2_metrics = (
    merged.groupby("agent")
    .agg(
        test_files_modified=("is_test_file", "sum"),
        test_loc_added=("test_loc_added", "sum"),
        test_loc_removed=("test_loc_removed", "sum"),
        total_assertions=("assertion_count", "sum")
    )
    .reset_index()
    .merge(coverage_proxy[["agent", "test_to_code_ratio"]], on="agent")
)

rq2_metrics["assertions_per_test_file"] = (
    rq2_metrics["total_assertions"] /
    rq2_metrics["test_files_modified"].replace(0, 1)
)

display(rq2_metrics)


,agent,test_files_modified,test_loc_added,test_loc_removed,total_assertions,test_to_code_ratio,assertions_per_test_file
0,Claude_Code,1641,104998,75449,19585,0.079727,11.934796
1,Copilot,17674,469167,328639,31085,0.106953,1.758798
2,Cursor,1405,104272,45157,11771,0.076997,8.377936
3,Devin,11618,397619,172855,58289,0.096849,5.017129
4,OpenAI_Codex,5674,233540,42753,23234,0.030863,4.094818


In [20]:
rq2_metrics.to_excel("RQ2_agentic_ai_test_metrics.xlsx", index=False)
merged.to_csv("RQ2_detailed_patch_metrics.csv", index=False, encoding="utf-8")

print("Saved RQ2 results.")

Saved RQ2 results.
